In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1: imports, HF login, transforms, model_configs (ALL your code)
# ─────────────────────────────────────────────────────────────────────────────

import os
import sys
import re
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.cluster import KMeans
import torch
import h5py

# PATH = "../.."
PATH = "storage"

# ensure EXAONEPath VT is on PYTHONPATH
github_path = os.path.join(PATH, "third_party", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer

# HuggingFace login
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Please set HF_TOKEN in your environment")
login(token=hf_token, add_to_git_credential=True)

# For transforms
from torchvision import transforms
import timm
from timm.data import resolve_data_config, create_transform
from transformers import AutoImageProcessor, AutoModel, AutoModelForImageClassification

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# ProcessorTransform for HF pipelines
class ProcessorTransform:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, img):
        return self.processor(img, return_tensors="pt")["pixel_values"].squeeze(0)

# Torch/timm model transforms
uni_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
giga_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
exaone_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
hoptimus_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517))
])
virchow_transform = transforms.Compose([
    transforms.Resize(size=224, interpolation=transforms.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4850, 0.4560, 0.4060), std=(0.2290, 0.2240, 0.2250))
])

# HF processors
resnet_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
hibou_b_processor   = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True)
hibou_L_processor   = AutoImageProcessor.from_pretrained("histai/hibou-L", trust_remote_code=True)
phikon_processor    = AutoImageProcessor.from_pretrained("owkin/phikon", trust_remote_code=True)
phikonv2_processor  = AutoImageProcessor.from_pretrained("owkin/phikon-v2", trust_remote_code=True)

resnet_transform = ProcessorTransform(resnet_processor)
hibou_b_transform  = ProcessorTransform(hibou_b_processor)
hibou_L_transform  = ProcessorTransform(hibou_L_processor)
phikon_transform   = ProcessorTransform(phikon_processor)
phikonv2_transform = ProcessorTransform(phikonv2_processor)

model_transforms = {
    "ResNet18":     resnet_transform,
    "ResNet50":     resnet_transform,
    "UNI":          uni_transform,
    "UNI2":         uni_transform,
    "GigaPath":     giga_transform,
    "Virchow":      virchow_transform,
    "Virchow2":     virchow_transform,
    "EXAONEPath":   exaone_transform,
    "Hibou-B":      hibou_b_transform,
    "Hibou-L":      hibou_L_transform,
    "H-Optimus-0":  hoptimus_transform,
    "H-Optimus-1":  hoptimus_transform,
    "H0-mini":      hoptimus_transform,
    "Phikon v1":    phikon_transform,
    "Phikon v2":    phikonv2_transform,
}

def timm_transform(model):
    if hasattr(model, "pretrained_cfg"):
        timm_model = model
    elif hasattr(model, "backbone") and hasattr(model.backbone, "pretrained_cfg"):
        timm_model = model.backbone
    else:
        raise AttributeError("Cannot find .pretrained_cfg on the model or its backbone.")
    cfg = resolve_data_config(timm_model.pretrained_cfg, model=timm_model)
    return create_transform(**cfg)

def get_transform(model_name, model=None):
    if model_name in {"Virchow", "Virchow2", "H0-mini"} and model is not None:
        return timm_transform(model)
    return model_transforms[model_name]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [ ]:
# model_configs exactly as you provided
model_configs = [
    {
        "name": "ResNet18",
        "enabled": True,
        "init": lambda: AutoModelForImageClassification.from_pretrained("microsoft/resnet-18").eval().to("cuda"),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).logits.detach().cpu().numpy().squeeze(),
    },
    {
        "name": "ResNet50",
        "enabled": True,
        "init": lambda: timm.create_model("resnet50", pretrained=True).eval().to("cuda"),
        "transform": resnet_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "UNI",
        "enabled": True,
        "init": lambda: timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True).eval().to("cuda"),
        "transform": uni_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "UNI2",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:MahmoodLab/UNI2-h",
            pretrained=True,
            img_size=224,
            patch_size=14,
            depth=24,
            num_heads=24,
            init_values=1e-5,
            embed_dim=1536,
            mlp_ratio=2.66667*2,
            num_classes=0,
            no_embed_class=True,
            mlp_layer=timm.layers.SwiGLUPacked,
            act_layer=torch.nn.SiLU,
            reg_tokens=8,
            dynamic_img_size=True,
        ).eval().to("cuda"),
        "transform": uni_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "GigaPath",
        "enabled": True,
        "init": lambda: timm.create_model("hf_hub:prov-gigapath/prov-gigapath", pretrained=True).eval().to("cuda"),
        "transform": giga_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Virchow",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:paige-ai/Virchow",
            pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked,
            act_layer=torch.nn.SiLU,
        ).eval().to("cuda"),
        "transform": virchow_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to("cuda"))[:, 0],
            model(tensor.to("cuda"))[:, 1:].mean(1)
        ], dim=-1).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Virchow2",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:paige-ai/Virchow2",
            pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked,
            act_layer=torch.nn.SiLU,
        ).eval().to("cuda"),
        "transform": virchow_transform,
        "embedding_fn": lambda model, tensor: torch.cat([
            model(tensor.to("cuda"))[:, 0],
            model(tensor.to("cuda"))[:, 5:].mean(1)
        ], dim=-1).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "EXAONEPath",
        "enabled": True,
        "init": lambda: VisionTransformer.from_pretrained(
            "LGAI-EXAONE/EXAONEPath", use_auth_token=hf_token
        ).eval().cuda(),
        "transform": exaone_transform,
        "embedding_fn": lambda model, tensor: model(tensor.cuda()).squeeze().detach().cpu().numpy(),
    },
    {
        "name": "Hibou-B",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("histai/hibou-b", trust_remote_code=True).eval().to("cuda"),
        "transform": hibou_b_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Hibou-L",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("histai/hibou-L", trust_remote_code=True).eval().to("cuda"),
        "transform": hibou_L_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Phikon v1",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("owkin/phikon", trust_remote_code=True).eval().to("cuda"),
        "transform": phikon_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "Phikon v2",
        "enabled": True,
        "init": lambda: AutoModel.from_pretrained("owkin/phikon-v2", trust_remote_code=True).eval().to("cuda"),
        "transform": phikonv2_transform,
        "embedding_fn": lambda model, tensor: model(pixel_values=tensor.to("cuda")).last_hidden_state[:, 0, :].detach().cpu().numpy().squeeze(),
    },
    {
        "name": "H-Optimus-0",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H-optimus-0", pretrained=True, init_values=1e-5, dynamic_img_size=False
        ).eval().to("cuda"),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "H-Optimus-1",
        "enabled": True,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H-optimus-1", pretrained=True, init_values=1e-5, dynamic_img_size=False
        ).eval().to("cuda"),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
    {
        "name": "H0-mini",
        "enabled": False,
        "init": lambda: timm.create_model(
            "hf-hub:bioptimus/H0-mini", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU
        ).eval().to("cuda"),
        "transform": hoptimus_transform,
        "embedding_fn": lambda model, tensor: model(tensor.to("cuda")).detach().cpu().numpy().squeeze(),
    },
]

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 (configurable % per split): sampling parameters, I/O paths, and build per‐split sample sets
# ─────────────────────────────────────────────────────────────────────────────

# fraction of patches per WSI to sample, by split
sample_pct = {
    "train": 0.19,   # e.g. 10% of patches per WSI in train
    "test":  0.99,   # 20% in test
    "valid": 0.51,   # 15% in valid
}

# number of top‐WSIs by patch‐count to include
n_top_wsis = 15

# TSNE settings
tsne_perplexity    = 50
tsne_random_state  = 42

# Meta‐CSV & H5 templates
data_dir      = os.path.join(PATH, "data", "PatchCamelyon")
meta_template = os.path.join(data_dir, "images", "camelyonpatch_level_2_split_{set}_meta.csv")
h5_template   = os.path.join(data_dir, "images",
                             "camelyonpatch_level_2_split_{set}_x.h5")

# where to save embeddings & plots
out_base = os.path.join(data_dir, "set_similarity", "embeddings_comparison")
emb_dir  = os.path.join(out_base, "embeddings")
tsne_dir = os.path.join(out_base, "tsne")
os.makedirs(emb_dir,  exist_ok=True)
os.makedirs(tsne_dir, exist_ok=True)

import h5py

def sample_patches(set_name):
    pct = sample_pct.get(set_name, 0.10)  # default to 10% if missing
    meta = pd.read_csv(meta_template.format(set=set_name))
    # --- print stats ---
    print(f"\n=== {set_name.upper()} STATISTICS ===")
    print(f"Unique WSIs: {meta['wsi'].nunique()}")
    counts = meta.groupby("wsi").size().sort_values(ascending=False)
    print(f" Top {n_top_wsis} WSIs by patch count:\n", counts.head(n_top_wsis))
    if set_name in ["train","valid"]:
        normal = meta['wsi'].str.contains("normal", case=False).sum()
        tumor  = meta['wsi'].str.contains("tumor",  case=False).sum()
        print(f"Patches from 'normal': {normal}, from 'tumor': {tumor}")
    # --- sample per WSI ---
    top_wsis = counts.head(n_top_wsis).index.tolist()
    sampled = []
    for w in top_wsis:
        wdf = meta[meta["wsi"] == w]
        n = max(1, int(len(wdf) * pct))
        sampled.append(wdf.sample(n, random_state=0))
    sampled_meta = pd.concat(sampled, ignore_index=True)

    # --- load images from H5 one‐by‐one to preserve order ---
    idxs = sampled_meta["index"].values
    with h5py.File(h5_template.format(set=set_name), "r") as f:
        imgs = np.stack([f["x"][i] for i in idxs], axis=0)  # shape [N, H, W, C]

    return sampled_meta, imgs

# build datasets dict
datasets = {}
for split in ["train","test","valid"]:
    df_split, imgs_split = sample_patches(split)
    datasets[split] = {"meta": df_split, "imgs": imgs_split}


=== TRAIN STATISTICS ===
Unique WSIs: 216
 Top 15 WSIs by patch count:
 wsi
camelyon16_train_tumor_093    3800
camelyon16_train_tumor_098    3695
camelyon16_train_tumor_094    3581
camelyon16_train_tumor_087    3238
camelyon16_train_tumor_004    3217
camelyon16_train_tumor_096    3016
camelyon16_train_tumor_105    3012
camelyon16_train_tumor_061    2957
camelyon16_train_tumor_103    2880
camelyon16_train_tumor_107    2837
camelyon16_train_tumor_048    2764
camelyon16_train_tumor_102    2729
camelyon16_train_tumor_041    2720
camelyon16_train_tumor_083    2676
camelyon16_train_tumor_017    2607
dtype: int64
Patches from 'normal': 82424, from 'tumor': 179720

=== TEST STATISTICS ===
Unique WSIs: 129
 Top 15 WSIs by patch count:
 wsi
camelyon16_test_066    716
camelyon16_test_079    678
camelyon16_test_099    677
camelyon16_test_122    658
camelyon16_test_084    648
camelyon16_test_113    596
camelyon16_test_065    589
camelyon16_test_051    570
camelyon16_test_013    566
camelyon16_test

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3: compute & save embeddings, cosine‐similarities, and joint t‐SNEs
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

for cfg in model_configs:
    if not cfg["enabled"]:
        continue
    name  = cfg["name"]
    print(f"\n>>> MODEL: {name}")
    model = cfg["init"]()
    transform = cfg["transform"]

    embs_by_set = {}
    # 1) embed each split
    for split, data in datasets.items():
        meta_df, imgs = data["meta"], data["imgs"]
        embs = []
        for img_arr in tqdm(imgs, desc=f"{name} on {split}"):
            img = Image.fromarray(img_arr)
            inp = transform(img).unsqueeze(0)
            emb = cfg["embedding_fn"](model, inp)
            embs.append(emb)
        embs = np.stack(embs, axis=0)
        embs_by_set[split] = embs
        # save raw embeddings + meta
        np.save(os.path.join(emb_dir, f"{name}_{split}_emb.npy"), embs)
        meta_df.to_csv(os.path.join(emb_dir, f"{name}_{split}_meta.csv"), index=False)

    # 2) cosine‐similarity of mean embeddings
    means = {s: embs_by_set[s].mean(axis=0) for s in embs_by_set}
    sets  = list(means.keys())
    M = cosine_similarity([means[s] for s in sets])
    sim_df = pd.DataFrame(M, index=sets, columns=sets)
    print("Cosine similarity between set‐means:\n", sim_df)

    # 3) joint t-SNE
    all_emb = np.concatenate([embs_by_set[s] for s in sets], axis=0)
    tsne = TSNE(n_components=2, perplexity=tsne_perplexity,
                random_state=tsne_random_state)
    XY = tsne.fit_transform(all_emb)

    # plot colored by split
    offsets = np.cumsum([0] + [embs_by_set[s].shape[0] for s in sets])
    plt.figure(figsize=(8,6))
    colors = {"train":"tab:blue","test":"tab:green","valid":"tab:orange"}
    for i, s in enumerate(sets):
        st, ed = offsets[i], offsets[i+1]
        plt.scatter(XY[st:ed,0], XY[st:ed,1],
                    s=6, alpha=0.6, label=s, c=colors[s])
    plt.legend()
    plt.title(f"{name} patch‐embeddings t-SNE by dataset")
    plt.xlabel("t-SNE1"); plt.ylabel("t-SNE2")
    plt.tight_layout()
    out_fig = os.path.join(tsne_dir, f"{name}_tsne_datasets.png")
    plt.savefig(out_fig, dpi=200)
    plt.close()
    print(f"Saved t-SNE plot → {out_fig}")


>>> MODEL: ResNet18


ResNet18 on train:  84%|████████▍ | 7332/8684 [00:26<00:04, 278.60it/s]


KeyboardInterrupt: 